<a href="https://colab.research.google.com/github/AsfandyarFarooqi/learning-log/blob/main/week07_real_gnss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
np.set_printoptions(suppress=True)

In [ ]:
a = 6378137.0
f = 1 / 298.257223563
e2 = 2 * f - f**2
b = a * (1 - f)
print("f =", round(f, 10), "| e2 =", round(e2, 10), "| b =", round(b, 3), "m")

def geodetic_to_ecef(lat_deg, lon_deg, h):
    lat = np.radians(lat_deg)
    lon = np.radians(lon_deg)
    N = a / np.sqrt(1 - e2 * np.sin(lat)**2)
    X = (N + h) * np.cos(lat) * np.cos(lon)
    Y = (N + h) * np.cos(lat) * np.sin(lon)
    Z = (N * (1 - e2) + h) * np.sin(lat)
    return np.array([X, Y, Z])

test = geodetic_to_ecef(30, 60, 0)
print("test point 30 N, 60 E, h = 0 (m):", np.round(test, 1))
p = np.sqrt(test[0]**2 + test[1]**2)
print("p =", round(p, 1), "m | on the ellipse? (p/a)^2 + (Z/b)^2 =", round((p / a)**2 + (test[2] / b)**2, 10))

f = 0.0033528107 | e2 = 0.00669438 | b = 6356752.314 m
test point 30 N, 60 E, h = 0 (m): [2764128.3 4787610.7 3170373.7]
p = 5528256.6 m | on the ellipse? (p/a)^2 + (Z/b)^2 = 1.0


In [ ]:
truth_ecef = geodetic_to_ecef(37.692231, -122.0884199, 20.974)
print("phone truth in ECEF (m):", np.round(truth_ecef, 3))
print("distance from Earth's centre (km):", round(np.sqrt(np.sum(truth_ecef**2)) / 1000, 3))

phone truth in ECEF (m): [-2684506.844 -4281392.596  3878481.691]
distance from Earth's centre (km): 6370.206


In [ ]:
def enu_matrix(lat_deg, lon_deg):
    lat = np.radians(lat_deg)
    lon = np.radians(lon_deg)
    east = [-np.sin(lon), np.cos(lon), 0.0]
    north = [-np.sin(lat) * np.cos(lon), -np.sin(lat) * np.sin(lon), np.cos(lat)]
    up = [np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)]
    return np.array([east, north, up])

R_test = enu_matrix(30, 60)
print("R at 30 N, 60 E:\n", np.round(R_test, 4))
print("R @ R.T:\n", np.round(R_test @ R_test.T, 10) + 0.0)
print("move (0, 0, 100) as E, N, U:", np.round(R_test @ np.array([0, 0, 100]), 2))
print("move (100, 0, 0) as E, N, U:", np.round(R_test @ np.array([100, 0, 0]), 2))

R at 30 N, 60 E:
 [[-0.866  0.5    0.   ]
 [-0.25  -0.433  0.866]
 [ 0.433  0.75   0.5  ]]
R @ R.T:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
move (0, 0, 100) as E, N, U: [ 0.  86.6 50. ]
move (100, 0, 0) as E, N, U: [-86.6 -25.   43.3]


In [ ]:
wls_ecef = np.array([-2684512.903, -4281393.666, 3878486.752])
d_ecef = wls_ecef - truth_ecef
R_phone = enu_matrix(37.692231, -122.0884199)
d_enu = R_phone @ d_ecef
print("Google's answer minus truth, ECEF (m):", np.round(d_ecef, 2))
print("Google's answer minus truth, E, N, U (m):", np.round(d_enu, 2))
print("length in ECEF:", round(np.sqrt(np.sum(d_ecef**2)), 2), "m | length in ENU:", round(np.sqrt(np.sum(d_enu**2)), 2), "m")

Google's answer minus truth, ECEF (m): [-6.06 -1.07  5.06]
Google's answer minus truth, E, N, U (m): [-4.56  1.48  6.36]
length in ECEF: 7.97 m | length in ENU: 7.97 m


In [ ]:
sat_G10 = np.array([-5111739.857, -13974794.020, 22176382.208])
enu_G10 = R_phone @ (sat_G10 - truth_ecef)
r_G10 = np.sqrt(np.sum(enu_G10**2))
el_G10 = np.degrees(np.arcsin(enu_G10[2] / r_G10))
az_G10 = np.degrees(np.arctan2(enu_G10[0], enu_G10[1]))
print("G10 as E, N, U (km):", np.round(enu_G10 / 1000, 3))
print("range (km):", round(r_G10 / 1000, 3))
print("elevation:", round(el_G10, 2), "deg | azimuth:", round(az_G10, 2), "deg")

G10 as E, N, U (km): [ 3092.977  8669.569 18706.649]
range (km): 20848.661
elevation: 63.8 deg | azimuth: 19.63 deg


In [ ]:
sat_G2=np.array([-14916644.088,8381795.848,20772371.222])
enu_G2=R_phone@(sat_G2-truth_ecef)
r_G2=np.sqrt(np.sum(enu_G2**2))
el_G2=np.degrees(np.arcsin(enu_G2[2]/r_G2))
az_G2=np.degrees(np.arctan2(enu_G2[0],enu_G2[1]));
print("G2 as E, N, U (km):",np.round(enu_G2/1000,3))
print("range (km):",round(r_G2/1000,3))
print("elevation:",round(el_G2,2),"deg | azimuth:",round(az_G2,2),"deg")

G2 as E, N, U (km): [-17090.457  15954.903   6981.574]
range (km): 24400.513
elevation: 16.63 deg | azimuth: -46.97 deg


In [ ]:
svid = np.array([2, 8, 10, 18, 21, 23, 24, 27, 28, 32])
sats_ecef = np.array([[-14916644.088, 8381795.848, 20772371.222],
                      [-20987738.292, -1376793.804, 16379998.400],
                      [-5111739.857, -13974794.020, 22176382.208],
                      [4662547.027, -26046772.507, -304155.421],
                      [-15329996.983, 4873323.738, 21506733.093],
                      [8735732.507, -17141539.403, 18238905.248],
                      [14512036.477, -11225369.117, 18680721.785],
                      [-22804233.674, -12220838.375, 6640973.669],
                      [-6555214.310, -22205195.468, -13039139.727],
                      [-14858588.220, -20090152.111, 9473949.458]])
raw = np.array([24567440.915, 22397058.939, 20864402.974, 23472434.871, 23502523.242,
                22386395.921, 23822439.568, 21815989.505, 24951022.966, 20888952.079])
sat_clock = np.array([-166876.028, -42304.128, -15721.262, -122784.859, 44763.811,
                      19570.306, -93804.429, -10684.238, -2019.730, -166129.923])
iono = np.array([9.277, 6.805, 4.827, 10.145, 8.076, 7.199, 9.803, 6.232, 15.365, 4.905])
tropo = np.array([8.645, 4.165, 2.784, 6.487, 5.885, 4.434, 9.052, 3.519, 17.993, 2.706])

rho = raw + sat_clock - iono - tropo
print("satellites:", svid)
print("corrected pseudoranges (m):", np.round(rho, 3))

satellites: [ 2  8 10 18 21 23 24 27 28 32]
corrected pseudoranges (m): [24400546.965 22354743.841 20848674.101 23349633.38  23547273.092
 22405954.594 23728616.284 21805295.516 24948969.878 20722814.545]


In [ ]:
def gnss_3d(sats, rho, x0, stop=0.0001, max_rounds=10, show=True):
    x = np.array(x0, dtype=float)
    n = len(rho)
    for round_no in range(1, max_rounds + 1):
        diff = x[:3] - sats
        d0 = np.sqrt(np.sum(diff**2, axis=1))
        A = np.ones((n, 4))
        A[:, :3] = diff / d0.reshape(n, 1)
        drho = rho - (d0 + x[3])
        dx = np.linalg.solve(A.T @ A, A.T @ drho)
        x = x + dx
        size = np.sqrt(np.sum(dx**2))
        if show:
            print("round", round_no, "| size of move:", round(size, 3), "m | X, Y, Z, b:", np.round(x, 3))
        if size < stop:
            break
    return x

x_hat = gnss_3d(sats_ecef, rho, np.array([0.0, 0.0, 0.0, 0.0]))

round 1 | size of move: 7572777.281 m | X, Y, Z, b: [-3166453.962 -5042663.619  4524330.895  1192685.47 ]
round 2 | size of move: 1588721.412 m | X, Y, Z, b: [-2695811.359 -4299439.967  3892254.125    30519.894]
round 3 | size of move: 39641.377 m | X, Y, Z, b: [-2684501.162 -4281420.862  3878485.227       36.333]
round 4 | size of move: 21.552 m | X, Y, Z, b: [-2684494.743 -4281410.737  3878478.008       19.943]
round 5 | size of move: 0.0 m | X, Y, Z, b: [-2684494.743 -4281410.737  3878478.008       19.943]


In [ ]:
err_enu = R_phone @ (x_hat[:3] - truth_ecef)
print("your error E, N, U (m):", np.round(err_enu, 2), "| length:", round(np.sqrt(np.sum(err_enu**2)), 2), "m")
print("Google's error E, N, U (m):", np.round(d_enu, 2), "| length:", round(np.sqrt(np.sum(d_enu**2)), 2), "m")

your error E, N, U (m): [19.89 -8.38  4.82] | length: 22.12 m
Google's error E, N, U (m): [-4.56  1.48  6.36] | length: 7.97 m


Friday - Visualize The Globe (X,Y,Z and U,N,E)

In [ ]:
import numpy as np
np.set_printoptions(suppress=True)

# ---------- 1. The place P (change these three to try other places) ----------
lat_deg = 30.0   # φ: latitude of P in degrees (north is +, south is -)
lon_deg = 60.0   # λ: longitude of P in degrees (east is +, west is -)
h = 0.0          # height of P above the WGS84 ellipsoid, in metres

# ---------- 2. The WGS84 ellipsoid and Tuesday's function ----------
a = 6378137.0              # equator radius of the Earth, metres
f = 1 / 298.257223563      # flattening of the Earth
e2 = 2 * f - f**2          # squared eccentricity e², 0.00669438

def geodetic_to_ecef(lat_deg, lon_deg, h):
    lat = np.radians(lat_deg)
    lon = np.radians(lon_deg)
    N = a / np.sqrt(1 - e2 * np.sin(lat)**2)
    X = (N + h) * np.cos(lat) * np.cos(lon)
    Y = (N + h) * np.cos(lat) * np.sin(lon)
    Z = (N * (1 - e2) + h) * np.sin(lat)
    return np.array([X, Y, Z])

P = geodetic_to_ecef(lat_deg, lon_deg, h)   # P in ECEF (X, Y, Z), metres

# ---------- 3. Method 1: nudge P and watch which way it moves ----------
step_deg = 0.00001  # a tiny change of latitude or longitude, in degrees (about 1 m on the ground)
step_h = 1.0        # a tiny change of height, in metres

move_east = geodetic_to_ecef(lat_deg, lon_deg + step_deg, h) - P    # walk east: only λ grows
move_north = geodetic_to_ecef(lat_deg + step_deg, lon_deg, h) - P   # walk north: only φ grows
move_up = geodetic_to_ecef(lat_deg, lon_deg, h + step_h) - P        # go up: only h grows

east_1 = move_east / np.sqrt(np.sum(move_east**2))      # same direction, length 1
north_1 = move_north / np.sqrt(np.sum(move_north**2))
up_1 = move_up / np.sqrt(np.sum(move_up**2))

# ---------- 4. Method 2: the formulas that fill R ----------
lat = np.radians(lat_deg)   # φ in radians
lon = np.radians(lon_deg)   # λ in radians
east_2 = np.array([-np.sin(lon), np.cos(lon), 0.0])
north_2 = np.array([-np.sin(lat) * np.cos(lon), -np.sin(lat) * np.sin(lon), np.cos(lat)])
up_2 = np.array([np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)])

R = np.array([east_2, north_2, up_2])   # row 1 = east, row 2 = north, row 3 = up

# ---------- 5. Print everything ----------
print("P in ECEF (km):", np.round(P / 1000, 3))
print()
print("one nudge east moves P by (m):", np.round(move_east, 4))
print("one nudge north moves P by (m):", np.round(move_north, 4))
print("one nudge up moves P by (m):", np.round(move_up, 4))
print()
print("east  arrow, method 1 (nudge):", np.round(east_1, 5), "| method 2 (formula):", np.round(east_2, 5))
print("north arrow, method 1 (nudge):", np.round(north_1, 5), "| method 2 (formula):", np.round(north_2, 5))
print("up    arrow, method 1 (nudge):", np.round(up_1, 5), "| method 2 (formula):", np.round(up_2, 5))
print()
print("R (rows: east, north, up; columns: X, Y, Z):\n", np.round(R, 5))
print()
d = np.array([100.0, 0.0, 0.0])   # an example arrow: 100 m along the X axis
print("d =", d, "-> R @ d = (dE, dN, dU):", np.round(R @ d, 2))

P in ECEF (km): [2764.128 4787.611 3170.374]

one nudge east moves P by (m): [-0.8356  0.4824  0.    ]
one nudge north moves P by (m): [-0.2771 -0.48    0.96  ]
one nudge up moves P by (m): [0.433 0.75  0.5  ]

east  arrow, method 1 (nudge): [-0.86603  0.5      0.     ] | method 2 (formula): [-0.86603  0.5      0.     ]
north arrow, method 1 (nudge): [-0.25    -0.43301  0.86603] | method 2 (formula): [-0.25    -0.43301  0.86603]
up    arrow, method 1 (nudge): [0.43301 0.75    0.5    ] | method 2 (formula): [0.43301 0.75    0.5    ]

R (rows: east, north, up; columns: X, Y, Z):
 [[-0.86603  0.5      0.     ]
 [-0.25    -0.43301  0.86603]
 [ 0.43301  0.75     0.5    ]]

d = [100.   0.   0.] -> R @ d = (dE, dN, dU): [-86.6 -25.   43.3]


In [ ]:
import plotly.graph_objects as go

km = 1000.0    # metres per kilometre: the picture is drawn in km
L = 3000.0     # drawn length of each direction arrow, km (only for the picture; the real arrows have length 1)

fig = go.Figure()

# ---- the globe: meridians (constant λ) and parallels (constant φ), drawn with your own function at h = 0 ----
for lon_line in range(-180, 180, 30):
    pts = np.array([geodetic_to_ecef(lat_line, lon_line, 0) for lat_line in np.linspace(-90, 90, 61)]) / km
    colour = "gray" if lon_line == 0 else "lightgray"     # Greenwich (λ = 0°) a bit darker
    fig.add_trace(go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode="lines",
                               line=dict(color=colour, width=2), showlegend=False, hoverinfo="skip"))
for lat_line in range(-60, 90, 30):
    pts = np.array([geodetic_to_ecef(lat_line, lon_line, 0) for lon_line in np.linspace(-180, 180, 121)]) / km
    colour = "gray" if lat_line == 0 else "lightgray"     # the equator (φ = 0°) a bit darker
    fig.add_trace(go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode="lines",
                               line=dict(color=colour, width=2), showlegend=False, hoverinfo="skip"))

# ---- a helper that draws one arrow: a line from start, plus a dot and a label at its tip ----
def draw_arrow(start, direction, length, colour, label, width=7, dash="solid"):
    end = start + length * direction
    fig.add_trace(go.Scatter3d(x=[start[0], end[0]], y=[start[1], end[1]], z=[start[2], end[2]],
                               mode="lines", line=dict(color=colour, width=width, dash=dash),
                               name=label, hoverinfo="skip"))
    fig.add_trace(go.Scatter3d(x=[end[0]], y=[end[1]], z=[end[2]], mode="markers+text",
                               marker=dict(size=4, color=colour), text=[label],
                               textposition="top center", showlegend=False, hoverinfo="skip"))

# ---- the ECEF axes X, Y, Z from the Earth's centre O ----
O = np.array([0.0, 0.0, 0.0])
axis_length = 1.4 * a / km
draw_arrow(O, np.array([1.0, 0.0, 0.0]), axis_length, "black", "X (equator, 0° longitude)")
draw_arrow(O, np.array([0.0, 1.0, 0.0]), axis_length, "black", "Y (equator, 90° E)")
draw_arrow(O, np.array([0.0, 0.0, 1.0]), axis_length, "black", "Z (North Pole)")

# ---- the place P and the line from O to P ----
P_km = P / km
fig.add_trace(go.Scatter3d(x=[0, P_km[0]], y=[0, P_km[1]], z=[0, P_km[2]], mode="lines",
                           line=dict(color="black", width=2, dash="dot"), name="O to P", hoverinfo="skip"))
fig.add_trace(go.Scatter3d(x=[P_km[0]], y=[P_km[1]], z=[P_km[2]], mode="markers+text",
                           marker=dict(size=6, color="black"), text=["P"], textposition="bottom center",
                           showlegend=False, hoverinfo="skip"))

# ---- copies of the X, Y, Z directions moved to P (thin, dashed): the "1 m along X, Y, Z" of the R columns ----
draw_arrow(P_km, np.array([1.0, 0.0, 0.0]), 0.6 * L, "gray", "X direction at P", width=3, dash="dash")
draw_arrow(P_km, np.array([0.0, 1.0, 0.0]), 0.6 * L, "gray", "Y direction at P", width=3, dash="dash")
draw_arrow(P_km, np.array([0.0, 0.0, 1.0]), 0.6 * L, "gray", "Z direction at P", width=3, dash="dash")

# ---- the three rows of R, drawn at P ----
draw_arrow(P_km, east_2, L, "#D55E00", "east (row 1 of R)")
draw_arrow(P_km, north_2, L, "#0072B2", "north (row 2 of R)")
draw_arrow(P_km, up_2, L, "#009E73", "up (row 3 of R)")

# ---- where the camera looks from: above P, a little to the east and south ----
eye = up_2 + 0.5 * east_2 - 0.4 * north_2
eye = 1.8 * eye / np.sqrt(np.sum(eye**2))
fig.update_layout(title=f"East, north and up at P (latitude {lat_deg}°, longitude {lon_deg}°)",
                  scene=dict(aspectmode="data", xaxis=dict(title="X (km)"), yaxis=dict(title="Y (km)"), zaxis=dict(title="Z (km)")),
                  scene_camera=dict(eye=dict(x=float(eye[0]), y=float(eye[1]), z=float(eye[2]))),
                  height=750)
fig.show()

##Saturday

In [1]:
import numpy as np
np.set_printoptions(suppress=True)

# --- the Earth (WGS84) and Tuesday's function ---
a = 6378137.0
f = 1 / 298.257223563
e2 = 2 * f - f**2

def geodetic_to_ecef(lat_deg, lon_deg, h):
    lat = np.radians(lat_deg)
    lon = np.radians(lon_deg)
    N = a / np.sqrt(1 - e2 * np.sin(lat)**2)
    X = (N + h) * np.cos(lat) * np.cos(lon)
    Y = (N + h) * np.cos(lat) * np.sin(lon)
    Z = (N * (1 - e2) + h) * np.sin(lat)
    return np.array([X, Y, Z])

# --- Friday's R matrix ---
def enu_matrix(lat_deg, lon_deg):
    lat = np.radians(lat_deg)
    lon = np.radians(lon_deg)
    east = [-np.sin(lon), np.cos(lon), 0.0]
    north = [-np.sin(lat) * np.cos(lon), -np.sin(lat) * np.sin(lon), np.cos(lat)]
    up = [np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)]
    return np.array([east, north, up])

truth_ecef = geodetic_to_ecef(37.692231, -122.0884199, 20.974)
R_phone = enu_matrix(37.692231, -122.0884199)

# --- Friday's data: second 1, the 10 GPS L1 satellites ---
svid = np.array([2, 8, 10, 18, 21, 23, 24, 27, 28, 32])
sats_ecef = np.array([[-14916644.088, 8381795.848, 20772371.222],
                      [-20987738.292, -1376793.804, 16379998.400],
                      [-5111739.857, -13974794.020, 22176382.208],
                      [4662547.027, -26046772.507, -304155.421],
                      [-15329996.983, 4873323.738, 21506733.093],
                      [8735732.507, -17141539.403, 18238905.248],
                      [14512036.477, -11225369.117, 18680721.785],
                      [-22804233.674, -12220838.375, 6640973.669],
                      [-6555214.310, -22205195.468, -13039139.727],
                      [-14858588.220, -20090152.111, 9473949.458]])
raw = np.array([24567440.915, 22397058.939, 20864402.974, 23472434.871, 23502523.242,
                22386395.921, 23822439.568, 21815989.505, 24951022.966, 20888952.079])
sat_clock = np.array([-166876.028, -42304.128, -15721.262, -122784.859, 44763.811,
                      19570.306, -93804.429, -10684.238, -2019.730, -166129.923])
iono = np.array([9.277, 6.805, 4.827, 10.145, 8.076, 7.199, 9.803, 6.232, 15.365, 4.905])
tropo = np.array([8.645, 4.165, 2.784, 6.487, 5.885, 4.434, 9.052, 3.519, 17.993, 2.706])
rho = raw + sat_clock - iono - tropo

# --- Friday's solver (one change: "size" is now called "magnitude") ---
def gnss_3d(sats, rho, x0, stop=0.0001, max_rounds=10, show=True):
    x = np.array(x0, dtype=float)
    n = len(rho)
    for round_no in range(1, max_rounds + 1):
        diff = x[:3] - sats
        d0 = np.sqrt(np.sum(diff**2, axis=1))
        A = np.ones((n, 4))
        A[:, :3] = diff / d0.reshape(n, 1)
        drho = rho - (d0 + x[3])
        dx = np.linalg.solve(A.T @ A, A.T @ drho)
        x = x + dx
        magnitude = np.sqrt(np.sum(dx**2))
        if show:
            print("round", round_no, "| magnitude of move:", round(magnitude, 3), "m | X, Y, Z, b:", np.round(x, 3))
        if magnitude < stop:
            break
    return x

x_hat = gnss_3d(sats_ecef, rho, np.array([0.0, 0.0, 0.0, 0.0]), show=False)
err = R_phone @ (x_hat[:3] - truth_ecef)
print("Friday's answer X, Y, Z, b:", np.round(x_hat, 3))
print("Friday's error E, N, U (m):", np.round(err, 2), "| magnitude:", round(np.sqrt(np.sum(err**2)), 2), "m")

Friday's answer X, Y, Z, b: [-2684494.743 -4281410.737  3878478.008       19.943]
Friday's error E, N, U (m): [19.89 -8.38  4.82] | magnitude: 22.12 m


In [2]:
omega = 0.000072921151467   # the Earth's turn rate (rad/s), WGS84
c = 299792458.0             # the speed of light (m/s)

def turn_sats(sats, x):
    r = np.sqrt(np.sum((sats - x[:3])**2, axis=1))
    tau = r / c
    theta = omega * tau
    X = sats[:, 0]
    Y = sats[:, 1]
    X_new = X * np.cos(theta) + Y * np.sin(theta)
    Y_new = -X * np.sin(theta) + Y * np.cos(theta)
    return np.column_stack([X_new, Y_new, sats[:, 2]])

sats_fixed = turn_sats(sats_ecef, x_hat)
print("G10 before:", np.round(sats_ecef[2], 3))
print("G10 after: ", np.round(sats_fixed[2], 3))

x_fix = gnss_3d(sats_fixed, rho, x_hat)
err_fix = R_phone @ (x_fix[:3] - truth_ecef)
print("fixed error E, N, U (m):", np.round(err_fix, 2), "| magnitude:", round(np.sqrt(np.sum(err_fix**2)), 2), "m")

G10 before: [ -5111739.857 -13974794.02   22176382.208]
G10 after:  [ -5111810.726 -13974768.097  22176382.208]
round 1 | magnitude of move: 28.34 m | X, Y, Z, b: [-2684518.465 -4281395.238  3878478.487       20.001]
round 2 | magnitude of move: 0.0 m | X, Y, Z, b: [-2684518.465 -4281395.238  3878478.487       20.001]
fixed error E, N, U (m): [-8.44 -7.68  4.7 ] | magnitude: 12.34 m


In [3]:
sats_fixed2 = turn_sats(sats_ecef, x_fix)
x_fix2 = gnss_3d(sats_fixed2, rho, x_fix)
change = x_fix2[:3] - x_fix[:3]
print("change from the second pass (mm):", round(np.sqrt(np.sum(change**2)) * 1000, 3))

round 1 | magnitude of move: 0.0 m | X, Y, Z, b: [-2684518.465 -4281395.238  3878478.487       20.001]
change from the second pass (mm): 0.016


In [12]:
##how far each satellite moved
sats_moved=sats_ecef-sats_fixed
magnitude_sats=np.sqrt(np.sum(sats_moved**2,axis=1))
time_sats=magnitude_sats/c

print(np.column_stack((svid,time_sats,magnitude_sats)))

[[  2.           0.00000034 101.55201197]
 [  8.           0.00000038 114.36684955]
 [ 10.           0.00000025  75.46124774]
 [ 18.           0.0000005  150.28491703]
 [ 21.           0.00000031  92.13403491]
 [ 23.           0.00000035 104.85333676]
 [ 24.           0.00000035 105.89291776]
 [ 27.           0.00000046 137.22438131]
 [ 28.           0.00000047 140.50260779]
 [ 32.           0.00000042 125.95334973]]
